# Colab 深度風格 Embedding 訓練與推論啟動器

對應 `aicup/train_player_emb.py` 與 `aicup/infer_player_emb.py`，checkpoint 與快取存到 Drive `MyDrive/aicup-2026/player-trained-models/`。

1. 執行階段選 GPU（建議 A100 或 L4，需高記憶體 RAM >= 30 GB）
2. 確認 Drive `MyDrive/aicup-2026/humansl-analysis/` 內有 `val-players-newdev-v180000.csv` 與 `val-players.csv`
3. 修改第一格 `MODE`（`smoke` / `full` / `stage2` / `model_b` / `infer_val` / `infer_b` / `infer_a`）與 `BRANCH`；`stage2` 需填 `RESUME_CKPT`（既有 `last.pt`），`infer_*` 需填 `INFER_CKPT`（既有 `best_model.pth`）；`HEAD_MOVES = 249` 時 `infer_*` 改產生前段視角快取 `*_head_emb.npz`
4. 執行階段 → 全部執行

In [ ]:
MODE = 'smoke'  # 'smoke', 'full', 'stage2', 'model_b', 'infer_val', 'infer_hold', 'infer_b', 'infer_a'
BRANCH = 'main'  # 可改為實驗分支名稱（例如 'exp/p07-emb'）避免動到 main
FEAT_DIR = 'rank-features-v180000'
INIT_CKPT = 'rank-trained-models/59844b9/best_model.pth'
RESUME_CKPT = ''  # stage2 模式填寫既有 last.pt，例如 'player-trained-models/emb-<hash>/last.pt'
INFER_CKPT = ''  # infer_* 模式填寫 best_model.pth，例如 'player-trained-models/emb-<hash>/best_model.pth'
INFER_TTA_A = 4  # A 組推論 TTA 視角數（4 或 8）
HEAD_MOVES = 0  # infer_* 前段視角填 249（加根節點空盤正好 250 個盤面），0 為完整視角
EXTRA_ARGS = []
AUTO_UNASSIGN = True

VALID_MODES = ('smoke', 'full', 'stage2', 'model_b', 'infer_val', 'infer_hold', 'infer_b', 'infer_a')
if MODE not in VALID_MODES:
    raise SystemExit('MODE 必須為 %s' % (VALID_MODES,))
if MODE == 'stage2' and not RESUME_CKPT:
    raise SystemExit('stage2 模式請先填寫 RESUME_CKPT')
if MODE.startswith('infer_') and not INFER_CKPT:
    raise SystemExit('推論模式請先填寫 INFER_CKPT')

In [ ]:
import os, shutil, subprocess, sys
from google.colab import drive, userdata

drive.mount('/content/drive')
REPO = '/content/aicup-2026-go'
DRIVE_ROOT = '/content/drive/MyDrive/aicup-2026'


def git(*args):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if r.returncode:
        print(r.stderr.replace(userdata.get('GH_TOKEN'), '***'))
        raise SystemExit('git %s 失敗' % args[0])
    return r.stdout.strip()


if not os.path.exists(REPO):
    git('clone', '-q', 'https://%s@github.com/Jx-study/aicup-2026-go.git' % userdata.get('GH_TOKEN'), REPO)
os.chdir(REPO)
git('fetch', '-q', 'origin')
git('checkout', '-q', BRANCH)
git('pull', '-q', 'origin', BRANCH)
print(git('log', '-1', '--oneline'))

# 資料放 GPU 或 CPU 由 train_player_emb.py 自行判斷，不需逐片統計大小
p = subprocess.Popen([sys.executable, '-u', '-m', 'aicup.colab_setup', '--feat-dir', FEAT_DIR, '--skip-size-check'],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end='')
if p.wait():
    raise SystemExit('colab_setup 失敗')

# 從 Drive 複製玩家切分清單，確保 dev 評估不混入 val648
os.makedirs('humansl-analysis', exist_ok=True)
for csv_name in ('val-players-newdev-v180000.csv', 'val-players.csv'):
    src = os.path.join(DRIVE_ROOT, 'humansl-analysis', csv_name)
    if not os.path.exists(src):
        raise SystemExit('Drive 缺少 %s，請先從本機上傳至 MyDrive/aicup-2026/humansl-analysis/' % src)
    shutil.copy2(src, os.path.join('humansl-analysis', csv_name))
print('已複製 dev 與 val648 玩家清單')

# 確保 feat_dir 下有 hd.npz（讓子局黑白反轉對照表），可從 rank-features 複用或由 training-v1.1 產生
hd_dst = os.path.join(FEAT_DIR, 'hd.npz')
if not os.path.exists(hd_dst):
    hd_fallback = os.path.join(DRIVE_ROOT, 'rank-features', 'hd.npz')
    if os.path.exists(hd_fallback):
        shutil.copy2(hd_fallback, hd_dst)
        print('已從 rank-features 複製 hd.npz 到', hd_dst)
    else:
        v11_dir = os.path.join(DRIVE_ROOT, 'training-v1.1', 'training')
        if not os.path.isdir(v11_dir):
            v11_dir = os.path.join(DRIVE_ROOT, 'training-v1.1')
        print('建立 hd.npz:', v11_dir, '->', hd_dst)
        subprocess.run([sys.executable, '-m', 'aicup.color', '--train-dir', v11_dir, '--out', hd_dst], check=True)

# 連結 player-trained-models 到 Drive
p_dir = os.path.join(DRIVE_ROOT, 'player-trained-models')
os.makedirs(p_dir, exist_ok=True)
if os.path.islink('player-trained-models'):
    os.unlink('player-trained-models')
if not os.path.exists('player-trained-models'):
    os.symlink(p_dir, 'player-trained-models')
print('player-trained-models ->', p_dir)

ram_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024 ** 3
print('系統 RAM %.1f GB' % ram_gb)
!nproc
!nvidia-smi

In [ ]:
HASH = git('rev-parse', '--short', 'HEAD')


def run_cmd(cmd):
    print('執行:', ' '.join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait():
        raise SystemExit('指令失敗，exit code %d' % p.returncode)


if MODE == 'smoke':
    CKPT_DIR = 'player-trained-models/emb-smoke-' + HASH
    cmd = [
        sys.executable, '-u', '-m', 'aicup.train_player_emb',
        '--feat-dir', FEAT_DIR,
        '--init', INIT_CKPT,
        '--ckpt-dir', CKPT_DIR,
        '--epochs', '2',
        '--steps-per-epoch', '1500',
        '--color', 'swap',
        *EXTRA_ARGS,
    ]
    run_cmd(cmd)
elif MODE == 'full':
    CKPT_DIR = 'player-trained-models/emb-' + HASH
    cmd = [
        sys.executable, '-u', '-m', 'aicup.train_player_emb',
        '--feat-dir', FEAT_DIR,
        '--init', INIT_CKPT,
        '--ckpt-dir', CKPT_DIR,
        '--epochs', '20',
        '--steps-per-epoch', '1500',
        '--color', 'swap',
        *EXTRA_ARGS,
    ]
    run_cmd(cmd)
elif MODE == 'stage2':
    CKPT_DIR = 'player-trained-models/emb-%s-s2' % HASH
    cmd = [
        sys.executable, '-u', '-m', 'aicup.train_player_emb',
        '--feat-dir', FEAT_DIR,
        '--resume-weights', RESUME_CKPT,
        '--ckpt-dir', CKPT_DIR,
        '--epochs', '15',
        '--steps-per-epoch', '1500',
        '--lr', '3e-4',
        '--backbone-lr', '1e-4',
        '--tail-crop', '20',
        '--seed', '142',
        '--color', 'swap',
        *EXTRA_ARGS,
    ]
    run_cmd(cmd)
elif MODE == 'model_b':
    CKPT_DIR = 'player-trained-models/emb-%s-b' % HASH
    cmd = [
        sys.executable, '-u', '-m', 'aicup.train_player_emb',
        '--feat-dir', FEAT_DIR,
        '--init', INIT_CKPT,
        '--ckpt-dir', CKPT_DIR,
        '--embed-dim', '512',
        '--tail-crop', '40',
        '--seed', '1024',
        '--epochs', '25',
        '--steps-per-epoch', '1500',
        '--color', 'swap',
        *EXTRA_ARGS,
    ]
    run_cmd(cmd)
elif MODE == 'infer_val':
    CACHE_DIR = os.path.join(os.path.dirname(INFER_CKPT), 'emb-cache')
    V11_DIR = os.path.join(DRIVE_ROOT, 'training-v1.1', 'training')
    if not os.path.isdir(V11_DIR):
        V11_DIR = os.path.join(DRIVE_ROOT, 'training-v1.1')
    HEAD_ARGS = ['--head-moves', str(HEAD_MOVES), '--sgf-dir', V11_DIR] if HEAD_MOVES else []
    SUFFIX = '_head_emb.npz' if HEAD_MOVES else '_emb.npz'
    for tag, p_csv in (
        ('newdev', 'humansl-analysis/val-players-newdev-v180000.csv'),
        ('val648', 'humansl-analysis/val-players.csv'),
    ):
        run_cmd([
            sys.executable, '-u', '-m', 'aicup.infer_player_emb',
            '--mode', 'packed',
            '--ckpt', INFER_CKPT,
            '--feat-dir', FEAT_DIR,
            '--players', p_csv,
            '--tta', '8',
            '--batch-size', '128',
            '--out', os.path.join(CACHE_DIR, tag + SUFFIX),
            *HEAD_ARGS,
            *EXTRA_ARGS,
        ])
elif MODE == 'infer_hold':
    CACHE_DIR = os.path.join(os.path.dirname(INFER_CKPT), 'emb-cache')
    HOLD_CSV = os.path.join('humansl-analysis', 'holdout-v180000.csv')
    src = os.path.join(DRIVE_ROOT, HOLD_CSV)
    if not os.path.exists(src):
        raise SystemExit('Drive 缺少 %s，請先從本機上傳' % src)
    shutil.copy2(src, HOLD_CSV)
    # TTA 同 infer_val，hold 與 newdev 的 embedding 才能比較
    run_cmd([
        sys.executable, '-u', '-m', 'aicup.infer_player_emb',
        '--mode', 'packed',
        '--ckpt', INFER_CKPT,
        '--feat-dir', FEAT_DIR,
        '--players', HOLD_CSV,
        '--tta', '8',
        '--batch-size', '128',
        '--out', os.path.join(CACHE_DIR, 'hold_emb.npz'),
        *EXTRA_ARGS,
    ])
elif MODE == 'infer_b':
    CACHE_DIR = os.path.join(os.path.dirname(INFER_CKPT), 'emb-cache')
    run_cmd([
        sys.executable, '-u', '-m', 'aicup.infer_player_emb',
        '--mode', 'test',
        '--ckpt', INFER_CKPT,
        '--groups', 'b',
        '--tta', '8',
        '--batch-size', '64',
        '--out-dir', CACHE_DIR,
        *(['--head-moves', str(HEAD_MOVES)] if HEAD_MOVES else []),
        *EXTRA_ARGS,
    ])
elif MODE == 'infer_a':
    CACHE_DIR = os.path.join(os.path.dirname(INFER_CKPT), 'emb-cache')
    run_cmd([
        sys.executable, '-u', '-m', 'aicup.infer_player_emb',
        '--mode', 'test',
        '--ckpt', INFER_CKPT,
        '--groups', 'a',
        '--tta', str(INFER_TTA_A),
        '--batch-size', '64',
        '--out-dir', CACHE_DIR,
        *(['--head-moves', str(HEAD_MOVES)] if HEAD_MOVES else []),
        *EXTRA_ARGS,
    ])

In [ ]:
if AUTO_UNASSIGN:
    from google.colab import runtime
    runtime.unassign()